# 05 Compare the single-view methods

Only reads the CSV files in `results/` (written by 01, 02, 03 and 04). Rerun a method notebook to update its
numbers. The extra analyses are in 06 and 07, multi-view is in `08_multiview`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = Path("results")

summary_files = [f for f in sorted(RESULTS_DIR.glob("*.csv")) if not f.stem.endswith("_per_user")]
results = pd.concat([pd.read_csv(f) for f in summary_files], ignore_index=True)
per_user = pd.concat([pd.read_csv(RESULTS_DIR / f"{f.stem}_per_user.csv") for f in summary_files], ignore_index=True)

print("methods found:", ", ".join(results["method"].unique()))

methods found: cure, kmeans, no_clustering, popularity


In [2]:
COLORS = {"cure": "tab:blue", "kmeans": "tab:orange", "no_clustering": "black", "popularity": "grey"}
REFERENCES = {"no_clustering": "--", "popularity": ":"}      # one run each, drawn as horizontal lines
METHODS = [m for m in COLORS if m in set(results["method"])]
CLUSTER_METHODS = [m for m in METHODS if m not in REFERENCES]
K_VALUES = sorted(results.loc[results["method"].isin(CLUSTER_METHODS), "k"].unique())


def k_axis(ax):
    ax.set_xscale("log")
    ax.set_xticks(K_VALUES)
    ax.set_xticklabels(K_VALUES)
    ax.minorticks_off()
    ax.set_xlabel("number of clusters k")


def reference_lines(ax, column, scale=1.0, which=REFERENCES):
    for method in which:
        if method in METHODS:
            value = results.loc[results["method"] == method, column].iloc[0] * scale
            ax.axhline(value, linestyle=REFERENCES[method], color=COLORS[method], label=method.replace("_", " "))

## All results

In [3]:
columns = ["method", "k", "clusters", "largest cluster %", "avg neighbour pool", "P@5", "P@10", "R@10",
           "P@20", "R@20", "clustering s", "CF s (all users, estimated)"]
results[columns].sort_values(["method", "k"]).round(4)

,method,k,clusters,largest cluster %,avg neighbour pool,P@5,P@10,R@10,P@20,R@20,clustering s,"CF s (all users, estimated)"
0,cure,10,9,34.9316,99910.9724,0.1794,0.1257,0.1257,0.0825,0.1650,33.1959,1179.3584
1,cure,20,14,32.9929,81934.1852,0.1756,0.1228,0.1228,0.0809,0.1618,33.3108,1080.5255
2,cure,40,24,31.0462,71129.9358,0.1719,0.1202,0.1202,0.0799,0.1598,33.5145,1054.7924
3,cure,80,46,27.5588,51653.9578,0.1628,0.1153,0.1153,0.0768,0.1536,33.8570,941.3008
4,kmeans,10,10,21.3061,63939.8329,0.1736,0.1226,0.1226,0.0808,0.1616,0.9842,1090.0598
5,kmeans,20,20,10.8641,28671.8298,0.1643,0.1160,0.1160,0.0772,0.1544,2.0097,865.6830
6,kmeans,40,40,6.2176,15523.7716,0.1545,0.1094,0.1094,0.0735,0.1471,3.2373,777.1501
7,kmeans,80,80,3.3510,8001.4138,0.1440,0.1025,0.1025,0.0694,0.1389,6.7962,733.3651
8,no_clustering,1,1,100.0000,476451.0000,0.1956,0.1356,0.1356,0.0881,0.1762,0.0000,2978.9322
9,popularity,1,1,100.0000,0.0000,0.0041,0.0041,0.0041,0.0037,0.0074,0.0000,582.6637


## Best k for every method

In [4]:
best = results.loc[results.groupby("method")["P@10"].idxmax()]
best = best.set_index("method").loc[METHODS].reset_index()
best[columns].round(4)

,method,k,clusters,largest cluster %,avg neighbour pool,P@5,P@10,R@10,P@20,R@20,clustering s,"CF s (all users, estimated)"
0,cure,10,9,34.9316,99910.9724,0.1794,0.1257,0.1257,0.0825,0.1650,33.1959,1179.3584
1,kmeans,10,10,21.3061,63939.8329,0.1736,0.1226,0.1226,0.0808,0.1616,0.9842,1090.0598
2,no_clustering,1,1,100.0000,476451.0000,0.1956,0.1356,0.1356,0.0881,0.1762,0.0000,2978.9322
3,popularity,1,1,100.0000,0.0000,0.0041,0.0041,0.0041,0.0037,0.0074,0.0000,582.6637
